In [ ]:
# Step 1: Install dependencies
# code for local tunnel
# !pip install -q streamlit scikit-learn seaborn pandas numpy openpyxl pyngrok

# code for ngrok
!pip install streamlit pyngrok scikit-learn seaborn pandas numpy openpyxl -q

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import seaborn as sns
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.svm import SVR, SVC

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# -------------------------------
# Page Config
# -------------------------------
st.set_page_config(page_title="ML App", layout="wide")
st.title("🤖 Machine Learning App")
st.write("Upload data or use an example dataset, select features & target, problem type, and then train models.")

# -------------------------------
# Initialize Session State
# -------------------------------
for key in ["trained_model", "scaler", "features_final", "model_name"]:
    if key not in st.session_state:
        st.session_state[key] = None

# -------------------------------
# Cache Dataset Loading
# -------------------------------
@st.cache_data
def load_dataset(file=None, dataset_name=None):
    if file:
        if file.name.endswith(".csv"):
            df = pd.read_csv(file)
        elif file.name.endswith(".xlsx"):
            df = pd.read_excel(file)
        else:
            df = pd.read_csv(file, sep="\t")
    else:
        df = sns.load_dataset(dataset_name)
    return df

# -------------------------------
# Sidebar: Data Source
# -------------------------------
st.sidebar.title("⚙️ Settings")
data_choice = st.sidebar.radio("Choose Data Source:", ["Upload Data", "Use Example Dataset"])

if data_choice == "Upload Data":
    uploaded_file = st.sidebar.file_uploader("Upload File", type=["csv", "xlsx", "tsv"])
    if uploaded_file:
        df = load_dataset(file=uploaded_file)
    else:
        st.stop()
else:
    dataset_name = st.sidebar.selectbox("Select Dataset", ["titanic", "tips", "iris"])
    df = load_dataset(dataset_name=dataset_name)

# -------------------------------
# Show basic data info
# -------------------------------
st.subheader("📊 Data Overview")
st.write(df.head())
st.write("Shape:", df.shape)
st.write("Columns:", df.columns.tolist())
st.write(df.describe(include='all'))

# -------------------------------
# Feature & Target Selection
# -------------------------------
st.subheader("🎯 Select Features and Target")
features = st.multiselect("Select Features", df.columns)
target = st.selectbox("Select Target", df.columns)

# -------------------------------
# Ask user for problem type
# -------------------------------
problem_type = st.radio("Problem Type", ["Regression", "Classification"])

# -------------------------------
# Sidebar: Model Selection (always visible)
# -------------------------------
st.sidebar.subheader("🤖 Model Selection")
if problem_type == "Regression":
    model_name = st.sidebar.selectbox(
        "Select Regression Model",
        ["Linear Regression", "Decision Tree", "Random Forest", "SVM"]
    )
    models = {
        "Linear Regression": LinearRegression(),
        "Decision Tree": DecisionTreeRegressor(),
        "Random Forest": RandomForestRegressor(),
        "SVM": SVR(),
    }
else:
    model_name = st.sidebar.selectbox(
        "Select Classification Model",
        ["Logistic Regression", "Decision Tree", "Random Forest", "SVM"]
    )
    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000),
        "Decision Tree": DecisionTreeClassifier(),
        "Random Forest": RandomForestClassifier(),
        "SVM": SVC(),
    }

st.session_state.model_name = model_name

# -------------------------------
# Button to start training
# -------------------------------
start_training = st.button("Start Training Models")

if start_training:
    if not features or not target:
        st.error("Please select features and target columns before training.")
        st.stop()

    X = df[features].copy()
    y = df[target].copy()

    # -------------------------------
    # Preprocessing
    # -------------------------------
    categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()

    encoders = {}
    for col in categorical_cols:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
        encoders[col] = le

    imputer = IterativeImputer()
    X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

    scaler = StandardScaler()
    X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

    if problem_type == "Classification" and y.dtype == "object":
        target_encoder = LabelEncoder()
        y = target_encoder.fit_transform(y.astype(str))

    split = st.slider("Train Size (%)", 50, 90, 80)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=(100 - split)/100, random_state=42
    )

    # -------------------------------
    # Train Model
    # -------------------------------
    model = models[st.session_state.model_name]
    model.fit(X_train, y_train)

    # Store trained model & scaler in session_state
    st.session_state.trained_model = model
    st.session_state.scaler = scaler
    st.session_state.features_final = features

    # -------------------------------
    # Evaluation
    # -------------------------------
    st.subheader("📈 Evaluation")
    y_pred = model.predict(X_test)
    if problem_type == "Regression":
        st.write({
            "MSE": mean_squared_error(y_test, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
            "MAE": mean_absolute_error(y_test, y_pred),
            "R2": r2_score(y_test, y_pred),
        })
    else:
        st.write({
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, average="weighted"),
            "Recall": recall_score(y_test, y_pred, average="weighted"),
            "F1": f1_score(y_test, y_pred, average="weighted"),
        })
        st.write("Confusion Matrix:")
        st.write(confusion_matrix(y_test, y_pred))

    st.success(f"✅ Trained Model: {model_name}")

    # -------------------------------
    # Download Model
    # -------------------------------
    if st.button("Download Model"):
        with open("model.pkl", "wb") as f:
            pickle.dump(model, f)
        st.success("Downloaded model.pkl")

# -------------------------------
# Prediction Section (only after training)
# -------------------------------
if st.session_state.trained_model and st.session_state.features_final:
    st.subheader("🔮 Prediction")
    input_data = []
    for col in st.session_state.features_final:
        val = st.number_input(f"Enter {col}", value=0.0)
        input_data.append(val)

    if st.button("Predict"):
        input_array = np.array(input_data).reshape(1, -1)
        input_array = st.session_state.scaler.transform(input_array)
        pred = st.session_state.trained_model.predict(input_array)
        st.success(f"Prediction: {pred[0]}")

# **`This code is for ngrok`**

In [ ]:
from pyngrok import ngrok

# Add your ngrok auth token
ngrok.set_auth_token("3BTL0BetbHJYAG3jTkMjiIV7chn_4D67RGRBxwDbGVggTjpmU")  # Get from https://dashboard.ngrok.com/get-started/your-authtoken

# Create stable public URL
public_url = ngrok.connect(8501)
print("🌍 Public URL:", public_url)

# Run Streamlit app
!streamlit run app.py &

# **`Belowe code is for Local tunnel`**

In [ ]:
# # Download local tunnal
# !npm install localtunnel

In [ ]:
# !pkill -f streamlit
# !pkill -f lt

In [ ]:
# # Run The App
# !streamlit run app.py &>/content/logs.txt &

In [ ]:
# # Load Online the App
# !npx localtunnel --port 8501